In [1]:
# Load the autoreload extension
%load_ext autoreload
# Set it to automatically reload all modules every time you run a cell
%autoreload 2

In [66]:
import torch
import tiktoken
import sys
sys.path.append('../scr')
from dataLoader import GPTDataset
from attentionLayers import MultiHeadMaskedAttention
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(42)

In [38]:
print(torch.__version__)

2.5.1


In [39]:
torch.cuda.is_available()

True

In [40]:
tokenizer = tiktoken.get_encoding("gpt2")

In [41]:
text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownplace."
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 5372, 13]


In [42]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownplace.


In [43]:
with open("../training_data/the-verdict.txt", "r") as f:
    raw_text = f.read()

In [44]:
dataset = GPTDataset(txt = raw_text, tokenizer=tokenizer, max_length=100, stride=10)

In [45]:
data_iter = iter(dataset)
first_batch = next(data_iter)

In [46]:
first_batch

(tensor([   40,   367,  2885,  1464,  1807,  3619,   402,   271, 10899,  2138,
           257,  7026, 15632,   438,  2016,   257,   922,  5891,  1576,   438,
           568,   340,   373,   645,  1049,  5975,   284,   502,   284,  3285,
           326,    11,   287,   262,  6001,   286,   465, 13476,    11,   339,
           550,  5710,   465, 12036,    11,  6405,   257,  5527, 27075,    11,
           290,  4920,  2241,   287,   257,  4489,    64,   319,   262, 34686,
         41976,    13,   357, 10915,   314,  2138,  1807,   340,   561,   423,
           587, 10598,   393, 28537,  2014,   198,   198,     1,   464,  6001,
           286,   465, 13476,     1,   438,  5562,   373,   644,   262,  1466,
          1444,   340,    13,   314,   460,  3285,  9074,    13, 46606,   536]),
 tensor([  367,  2885,  1464,  1807,  3619,   402,   271, 10899,  2138,   257,
          7026, 15632,   438,  2016,   257,   922,  5891,  1576,   438,   568,
           340,   373,   645,  1049,  5975,   284,

In [47]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True) # drop_last = True drops the last batch if it is shorter than the specified batch_size

In [48]:
dataloader_iter = iter(dataloader)

In [49]:
next(dataloader_iter)

[tensor([[ 8759,  2763,    26,  ...,   286,  2376, 26927],
         [ 2123, 10813,  1701,  ...,  1169,   691,  2134],
         [ 5365, 31655,    26,  ...,   616, 35957,    25],
         ...,
         [  438, 18108,   407,  ...,   683,   736,   284],
         [  517,    13,   383,  ...,     0,   198,   198],
         [  290,  1807,   683,  ...,  1762,    30,  2011]]),
 tensor([[ 2763,    26,   393,  ...,  2376, 26927,   616],
         [10813,  1701,   198,  ...,   691,  2134,  7163],
         [31655,    26,   475,  ..., 35957,    25,   366],
         ...,
         [18108,   407, 11196,  ...,   736,   284,   262],
         [   13,   383,  3200,  ...,   198,   198,     1],
         [ 1807,   683, 32081,  ...,    30,  2011, 29483]])]

In [67]:
d_in = 5
d_out = 4
context_length = 10
dropout = 0.2
num_heads = 2
qkv_bias = False
masked_attention = MultiHeadMaskedAttention(d_in = d_in, d_out = d_out, \
                                            context_length = context_length, \
                                            dropout = dropout, num_heads = num_heads, \
                                            qkv_bias = qkv_bias)

In [58]:
x = torch.rand(10, 5)

In [59]:
batch = torch.stack((x, x), dim = 0)

In [60]:
batch.shape

torch.Size([2, 10, 5])

In [61]:
atten_weights, outputs = masked_attention.forward(batch)

In [64]:
print(atten_weights.shape)
print(outputs.shape)

torch.Size([2, 10, 10])
torch.Size([2, 10, 3])


In [65]:
print(atten_weights[0])

tensor([[1.2500, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.6263, 0.6237, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.4169, 0.0000, 0.4167, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.3132, 0.3300, 0.3296, 0.2772, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.2544, 0.0000, 0.2426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.2118, 0.2136, 0.2153, 0.1892, 0.0000, 0.2099, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.1801, 0.1802, 0.0000, 0.1673, 0.1791, 0.1790, 0.1831, 0.0000, 0.0000,
         0.0000],
        [0.1559, 0.0000, 0.1564, 0.1538, 0.1559, 0.0000, 0.1630, 0.1506, 0.0000,
         0.0000],
        [0.0000, 0.1396, 0.1421, 0.1256, 0.0000, 0.0000, 0.1383, 0.1274, 0.1543,
         0.0000],
        [0.1279, 0.1255, 0.1261, 0.1203, 0.1269, 0.0000, 0.1231, 0.0000, 0.1342,
         0.0000]], grad_fn=<

In [ ]:
torch.random.seed()
dropout = torch.nn.Dropout(0.5)

In [56]:
# note that the remaining of the weights are times by 2
dropout(atten_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.9957, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.6908, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.4885, 0.5221, 0.4882, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.3833, 0.0000, 0.4221, 0.4029, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.3270, 0.0000, 0.3446, 0.0000, 0.3132, 0.0000, 0.0000, 0.0000,
         0.0000],
        [0.2819, 0.2887, 0.0000, 0.0000, 0.2862, 0.0000, 0.2799, 0.0000, 0.0000,
         0.0000],
        [0.0000, 0.0000, 0.2716, 0.2771, 0.0000, 0.2328, 0.2575, 0.2149, 0.0000,
         0.0000],
        [0.0000, 0.2163, 0.0000, 0.0000, 0.2283, 0.2094, 0.0000, 0.0000, 0.2129,
         0.0000],
        [0.0000, 0.0000, 0.0000, 0.1974, 0.2015, 0.0000, 0.0000, 0.1996, 0.1948,
         0.2025]], grad_fn=<